In [ ]:
import textwrap

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sentence_transformers import SentenceTransformer
from tabulate import tabulate
from tsnecuda import TSNE

In [ ]:
hkhs = pd.read_parquet("../data/interim/hkhs.parquet")
catalogue = pd.read_parquet("../data/interim/catalogue.parquet")

In [ ]:
smy = hkhs.query("field == 'description'") \
    [["chapter", "heading", "subheading", "hkhs"]] \
    .nunique() \
    .to_frame("HS Structure")
smy["Avg. Subclasses"] = (smy.shift(-1) / smy)

print(tabulate(smy.round(2), headers="keys", tablefmt="psql", showindex=False))

In [ ]:
dfs = []
for field in ["description", "commodity", "example"]:
    cnt = hkhs.query("field == @field")["desc"].value_counts()
    dfs.append(cnt.value_counts().to_frame(field))
cnt = pd.concat(dfs, axis=1).fillna(0).astype(int).reset_index(drop=True)

print(tabulate(cnt, headers="keys", tablefmt="psql", showindex=False))

In [ ]:
for field in ["commodity", "example"]:
    print(field, end="\n")
    tsne = hkhs.query("field == @field")
    for desc in tsne["desc"].value_counts().loc[lambda x: x > 1].index:
        subheadings = ", ".join(tsne.query("desc == @desc")["subheading"])
        desc = textwrap.shorten(desc, width=50, placeholder="...")
        print(f"> {subheadings} → {desc}", end="\n")

In [ ]:
hkhs["word_cnt"] = hkhs["desc"].fillna("").str.split().str.len()
plot_df = hkhs[hkhs["word_cnt"] > 0]

plt.figure(figsize=(8, 6))
sns.histplot(
    data=plot_df,
    x="word_cnt",
    hue="field",
    stat="proportion",
    common_norm=False,
    discrete=True,
    multiple="layer",
    alpha=0.3
)
plt.title("Distribution of Word Counts")
plt.xlabel("Number of words")
plt.ylabel("Proportion of observations")
plt.tight_layout()
plt.show()

In [ ]:
model = SentenceTransformer("BAAI/bge-small-en-v1.5", device="cuda")

tsne = hkhs.query("field == 'description'") \
    .merge(catalogue.rename(columns={"hsc": "chapter", "desc": "category"}))

embedding = model.encode(tsne["desc"].tolist())
X_embedded = TSNE(n_components=2, perplexity=15, learning_rate=10) \
    .fit_transform(embedding)

tsne = tsne.assign(
    x=X_embedded[:, 0],
    y=X_embedded[:, 1],
)

plt.figure(figsize=(8, 6))
plot = sns.scatterplot(
    x="x", y="y",
    hue="category",
    palette=sns.color_palette("hls", 22),
    data=tsne,
    alpha=0.3,
)
handles, labels = plt.gca().get_legend_handles_labels()
wrapped_labels = [
    textwrap.shorten(label, width=50, placeholder="...")
    for label in labels
]
plot.legend(
    handles=handles,
    labels=wrapped_labels,
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
    borderaxespad=0,
)
plt.title("Commodity Description Embedding")
plt.xlabel("")
plt.ylabel("")
plt.show()